## Grouping and Aggregation

Grouping and aggregation are core tools for summarizing data — instead of 
looking at individual rows one by one, we can split data into meaningful 
groups (e.g. by category, city, or status) and calculate a summary value 
for each group.

The general pattern is:
```python
df.groupby('column_name')['target_column'].agg_function()
```
| Function | What it does |
|----------|--------------|
| `sum()` | Total of all values in the group |
| `mean()` | Average value in the group |
| `median()` | Middle value — useful when data is skewed or has outliers |
| `count()` | Number of non-null entries in the group |
| `size()` | Number of rows in the group (includes nulls) |
| `min()` / `max()` | Smallest / largest value in the group |
| `std()` | Standard deviation — spread of values around the mean |
| `nunique()` | Number of unique values in the group |
| `agg([...])` | Apply multiple functions at once |
| `transform()` | Same aggregation, but returns a value for every row (not collapsed) — useful for filling missing values |

In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv('data/cleaned_ecommerce_data.csv')

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 650 entries, 0 to 649
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   order_id        650 non-null    str    
 1   customer_name   650 non-null    str    
 2   age             650 non-null    float64
 3   gender          650 non-null    str    
 4   city            650 non-null    str    
 5   order_date      650 non-null    str    
 6   product_name    650 non-null    str    
 7   category        650 non-null    str    
 8   unit_price      650 non-null    float64
 9   quantity        650 non-null    int64  
 10  email           650 non-null    str    
 11  payment_method  650 non-null    str    
 12  status          650 non-null    str    
dtypes: float64(2), int64(1), str(10)
memory usage: 66.1 KB


In [4]:
df['revenue'] = df['unit_price'] * df['quantity']
# Total sales by city
df.groupby('city')['revenue'].sum().astype(int)

city
Faisalabad    3785506
Gujranwala    3900733
Islamabad     4048839
Karachi       5330134
Lahore        3256200
Multan        3599600
Peshawar      6191040
Quetta        3643262
Rawalpindi    4837109
Sialkot       3520890
Unknown        446940
Name: revenue, dtype: int64

In [5]:
# Average orders by Payment Method
df.groupby('payment_method')['order_id'].count()

payment_method
Bank Transfer    130
Card             122
Cash             123
EasyPaisa        140
JazzCash         127
unknown            8
Name: order_id, dtype: int64

In [6]:
# What's the average spend of male vs female customers?
df.groupby('gender')['revenue'].mean()

gender
Female     66556.517418
Male       64469.040906
Unknown    61329.369000
Name: revenue, dtype: float64

In [7]:
# Which product category sold the most quantity?
df.groupby('category')['quantity'].sum().sort_values(ascending=False)

category
Accessories    466
Electronics    424
Office         131
Storage        112
Unknown          6
Name: quantity, dtype: int64

### Multiple Aggregations at once .agg()

In [8]:
df.groupby('city').agg({
    'revenue' : 'sum',
    'age' : 'mean',
    'order_id' : 'count'
})

,revenue,age,order_id
city,,,
Faisalabad,3.785507e+06,41.466667,60
Gujranwala,3.900733e+06,42.666667,66
Islamabad,4.048840e+06,40.354839,62
Karachi,5.330135e+06,42.562500,80
Lahore,3.256201e+06,41.519231,52
Multan,3.599600e+06,41.855072,69
Peshawar,6.191041e+06,39.157895,76
Quetta,3.643263e+06,40.428571,63
Rawalpindi,4.837110e+06,43.095238,63


### Grouping Multiple Columns & Aggregating Multiple Columns

In [16]:
df.groupby(['payment_method','gender'])['age'].mean()

payment_method  gender 
Bank Transfer   Female     41.439394
                Male       40.968254
                Unknown    24.000000
Card            Female     42.351852
                Male       40.530303
                Unknown    33.500000
Cash            Female     44.100000
                Male       42.866667
                Unknown    40.000000
EasyPaisa       Female     40.586667
                Male       40.258065
                Unknown    32.000000
JazzCash        Female     42.971831
                Male       41.727273
                Unknown    19.000000
unknown         Female     30.333333
                Male       43.200000
Name: age, dtype: float64

In [15]:
# Clean, readable column names in the output instead of the nested multi-level headers agg() normally produces.
df.groupby('payment_method')[['age','quantity']].mean()

,age,quantity
payment_method,,
Bank Transfer,41.076923,1.792308
Card,41.221311,1.672131
Cash,43.398374,1.845528
EasyPaisa,40.257143,1.635714
JazzCash,42.244094,1.795276
unknown,38.375000,2.250000


### Reset index

In [17]:
# After grouping, the group column becomes the index, not a regular column. 
# reset_index() turns it back into a normal column — handy the moment you want to sort, filter, or merge on it again.
# payment_method.reset_index()   -> but not needed here as i haven't stored it in the original df
df.head()

,order_id,customer_name,age,gender,city,order_date,product_name,category,unit_price,quantity,email,payment_method,status,revenue
0,ORD-10537,Maham Raza,60.0,Male,Islamabad,2025-02-06,Headphones,Accessories,9238.84,1,maham.raza29@gmail.com,EasyPaisa,Pending,9238.84
1,ORD-10016,Maham Sheikh,34.0,Female,Sialkot,2024-06-18,Smart Watch,Accessories,11754.36,4,maham.sheikh17@gmail.com,Bank Transfer,Pending,47017.44
2,ORD-10620,Hamza Sheikh,18.0,Female,Multan,2025-08-11,Monitor,Electronics,44713.92,3,hamza.sheikh90@gmail.com,EasyPaisa,Returned,134141.76
3,ORD-10254,Ahmed Ahmed,56.0,Male,Multan,2025-05-22,Keyboard,Accessories,4363.75,2,ahmed.ahmed33@gmail.com,Cash,Pending,8727.50
4,ORD-10527,Hina Butt,37.0,Male,Rawalpindi,2024-01-25,Monitor,Electronics,41632.93,2,hina.butt42@gmail.com,Bank Transfer,Returned,83265.86
